In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
import re


In [ ]:
KEYPOINT_DIR = '/content/drive/MyDrive/ISAS25/keypointlabel'

JOINTS = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

def clean_labels(df):
    df = df[df['Action Label'].notnull()].copy()
    df['Action Label'] = df['Action Label'].replace({'Throwing': 'Throwing things'})
    return df.reset_index(drop=True)

def normalize_pose_keypoints(df, joints=JOINTS):
    keypoints = np.zeros((len(df), len(joints), 2))
    for i, joint in enumerate(joints):
        keypoints[:, i, 0] = df[f'{joint}_x'].values
        keypoints[:, i, 1] = df[f'{joint}_y'].values
    mid_hip = (keypoints[:, joints.index('left_hip')] + keypoints[:, joints.index('right_hip')]) / 2
    keypoints_centered = keypoints - mid_hip[:, None, :]
    shoulder_vec = keypoints[:, joints.index('left_shoulder')] - keypoints[:, joints.index('right_shoulder')]
    shoulder_dist = np.linalg.norm(shoulder_vec, axis=1) + 1e-6
    keypoints_normalized = keypoints_centered / shoulder_dist[:, None, None]
    return keypoints_normalized


class KeypointAugmenter:
    def __init__(self, apply_prob=0.5):
        self.apply_prob = apply_prob

    def __call__(self, keypoints):
        if random.random() > self.apply_prob:
            return keypoints

        # Small random noise
        keypoints += np.random.normal(0, 0.01, keypoints.shape)

        # Random scaling
        scale = np.random.uniform(0.9, 1.1, (1, 1, 2))
        return keypoints * scale

def preprocess_keypoint_csv(df, augment=False):
    df = clean_labels(df)
    kp = normalize_pose_keypoints(df)

    if augment:
        augmenter = KeypointAugmenter()
        kp = np.array([augmenter(seg) for seg in kp])

    labels = df['Action Label'].values
    frame_ids = df['frame_id'].values
    return kp, labels, frame_ids

def load_all_keypoint_files(path):
    files = glob(os.path.join(path, 'keypoints_with_labels_*.csv'))
    all_data = []
    for f in files:
        match = re.search(r'keypoints_with_labels_(\d+)\.csv', os.path.basename(f))
        if match:
            user_id = int(match.group(1))
            df = pd.read_csv(f)
            df['user_id'] = user_id
            all_data.append(df)
    return all_data

def preprocess_all_users(dataframes):
    kp_all, label_all, id_all, frame_all = [], [], [], []
    for df in dataframes:
        kp, label, frame = preprocess_keypoint_csv(df)
        uid = df['user_id'].iloc[0]
        kp_all.append(kp)
        label_all.append(label)
        frame_all.append(frame)
        id_all.append([uid] * len(label))
    return kp_all, label_all, frame_all, id_all

dfs = load_all_keypoint_files(KEYPOINT_DIR)
kp_all, label_all, frame_all, id_all = preprocess_all_users(dfs)


In [ ]:
from collections import Counter, defaultdict

WINDOW_SIZE = 90
STRIDE = 15

def sliding_window_segments(keypoints, labels, frame_ids, window_size=WINDOW_SIZE, stride=STRIDE):
    segments, seg_labels, seg_frame_ids = [], [], []
    total_frames = keypoints.shape[0]
    for start in range(0, total_frames - window_size + 1, stride):
        end = start + window_size
        segment = keypoints[start:end]
        label_seq = labels[start:end]
        frame_seq = frame_ids[start:end]
        most_common_label = Counter(label_seq).most_common(1)[0][0]
        segments.append(segment)
        seg_labels.append(most_common_label)
        seg_frame_ids.append(frame_seq[0])
    return segments, seg_labels, seg_frame_ids

def segment_all_users(kp_all, label_all, frame_all, id_all):
    X, y, timestamps, user_ids = [], [], [], []
    for i in range(len(kp_all)):
        segments, labels, frames = sliding_window_segments(
            keypoints=kp_all[i],
            labels=label_all[i],
            frame_ids=frame_all[i]
        )
        X.extend(segments)
        y.extend(labels)
        timestamps.extend(frames)
        user_ids.extend(id_all[i][:len(segments)])
    return X, y, timestamps, user_ids

X_segments, y_segments, time_segments, user_segments = segment_all_users(kp_all, label_all, frame_all, id_all)

# Gom theo user để LOSO
user_data = defaultdict(lambda: {"X": [], "y": [], "uid": []})
for x, y, u in zip(X_segments, y_segments, user_segments):
    user_data[u]["X"].append(x)
    user_data[u]["y"].append(y)
    user_data[u]["uid"].append(u)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

# Build triplets
import random
def build_triplets(X_segments, y_segments, num_triplets=10000):
    label_to_indices = defaultdict(list)
    for i, label in enumerate(y_segments):
        label_to_indices[label].append(i)

    triplets = []
    labels = list(label_to_indices.keys())
    for _ in range(num_triplets):
        anchor_label = random.choice(labels)
        positive_indices = label_to_indices[anchor_label]
        anchor_idx, positive_idx = random.sample(positive_indices, 2)
        negative_label = random.choice([l for l in labels if l != anchor_label])
        negative_idx = random.choice(label_to_indices[negative_label])
        triplets.append((anchor_idx, positive_idx, negative_idx))
    return triplets

# Dataset for Triplet Loss
class TripletPoseDataset(Dataset):
    def __init__(self, segments, triplets):
        self.data = [torch.tensor(seg, dtype=torch.float32).view(90, -1) for seg in segments]
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]
        return self.data[a], self.data[p], self.data[n]

# Mô hình PoseEncoder dùng Transformer
class PoseEncoder(nn.Module):
    def __init__(self, input_dim=34, model_dim=64, emb_dim=128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=model_dim, nhead=4, batch_first=True),
            num_layers=3
        )
        self.bottleneck = nn.Linear(model_dim, emb_dim)

    def forward(self, x):  # x: [B, 90, 34]
        x = self.input_proj(x)
        x = self.encoder(x)
        z = self.bottleneck(x)
        return z.mean(dim=1)  # [B, 128]


In [ ]:
def train_encoder_triplet(model, dataset, epochs=10, batch_size=128, lr=1e-3, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for anchor, positive, negative in dataloader:
            anchor = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)

            z_a = model(anchor)
            z_p = model(positive)
            z_n = model(negative)

            loss = F.triplet_margin_loss(z_a, z_p, z_n, margin=1.0)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] Triplet Loss: {total_loss / len(dataloader):.4f}")

    return model

def extract_embeddings_from_encoder(segments, model, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model.eval()
    model.to(device)
    embeddings = []

    with torch.no_grad():
        for seg in segments:
            x = torch.tensor(seg, dtype=torch.float32).view(1, 90, -1).to(device)
            z = model(x)  # [1, 128]
            embeddings.append(z.cpu())

    return torch.cat(embeddings, dim=0)


In [ ]:
def run_loso_encoder_and_embedding(user_data, model_class, num_triplets=10000, epochs=10, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    all_embeddings = []
    all_labels = []
    all_user_ids = []

    for test_uid in user_data:
        print(f"\\n🔁 LOSO: Leaving out user {test_uid}")
        X_train, y_train = [], []
        for uid, d in user_data.items():
            if uid == test_uid:
                continue
            X_train.extend(d["X"])
            y_train.extend(d["y"])

        triplet_indices = build_triplets(X_train, y_train, num_triplets)
        triplet_dataset = TripletPoseDataset(X_train, triplet_indices)

        encoder = model_class()
        encoder = train_encoder_triplet(encoder, triplet_dataset, epochs=epochs, device=device)

        for uid, d in user_data.items():
            emb = extract_embeddings_from_encoder(d["X"], encoder, device=device)
            all_embeddings.append(emb)
            all_labels.extend(d["y"])
            all_user_ids.extend(d["uid"])

    all_embeddings = torch.cat(all_embeddings, dim=0)
    return all_embeddings, all_labels, all_user_ids

# === Gọi LOSO encoder pipeline ===
pose_embeddings_triplet, y_segments, user_segments = run_loso_encoder_and_embedding(
    user_data, PoseEncoder, num_triplets=10000, epochs=10
)
print("✅ Embedding shape:", pose_embeddings_triplet.shape)


\n🔁 LOSO: Leaving out user 1
[Epoch 1] Triplet Loss: 0.3564
[Epoch 2] Triplet Loss: 0.2303
[Epoch 3] Triplet Loss: 0.1857
[Epoch 4] Triplet Loss: 0.1422
[Epoch 5] Triplet Loss: 0.1216
[Epoch 6] Triplet Loss: 0.0921
[Epoch 7] Triplet Loss: 0.0912
[Epoch 8] Triplet Loss: 0.0718
[Epoch 9] Triplet Loss: 0.0524
[Epoch 10] Triplet Loss: 0.0726
\n🔁 LOSO: Leaving out user 3
[Epoch 1] Triplet Loss: 0.3269
[Epoch 2] Triplet Loss: 0.1971
[Epoch 3] Triplet Loss: 0.1505
[Epoch 4] Triplet Loss: 0.1137
[Epoch 5] Triplet Loss: 0.0858
[Epoch 6] Triplet Loss: 0.0788
[Epoch 7] Triplet Loss: 0.0670
[Epoch 8] Triplet Loss: 0.0534
[Epoch 9] Triplet Loss: 0.0511
[Epoch 10] Triplet Loss: 0.0403
\n🔁 LOSO: Leaving out user 2
[Epoch 1] Triplet Loss: 0.3319
[Epoch 2] Triplet Loss: 0.2149
[Epoch 3] Triplet Loss: 0.1657
[Epoch 4] Triplet Loss: 0.1307
[Epoch 5] Triplet Loss: 0.1072
[Epoch 6] Triplet Loss: 0.0894
[Epoch 7] Triplet Loss: 0.0825
[Epoch 8] Triplet Loss: 0.0648
[Epoch 9] Triplet Loss: 0.0578
[Epoch 10] T

In [ ]:
class MultiClassMILDataset(torch.utils.data.Dataset):
    def __init__(self, embeddings, labels, user_ids, segment_length=3):
        self.samples = []
        self.segment_length = segment_length
        self.label_names = sorted(set(labels))
        self.label_to_idx = {label: idx for idx, label in enumerate(self.label_names)}

        from collections import defaultdict
        user_to_idx = defaultdict(list)
        for i, uid in enumerate(user_ids):
            user_to_idx[uid].append((i, labels[i]))

        for uid, idx_label_pairs in user_to_idx.items():
            idx_seq = [i for i, _ in idx_label_pairs]
            labels_seq = [l for _, l in idx_label_pairs]

            for i in range(0, len(idx_seq) - segment_length + 1):
                segment_idxs = idx_seq[i:i + segment_length]
                middle_label = labels_seq[i + segment_length // 2]  # lấy label ở frame giữa
                label_idx = self.label_to_idx[middle_label]
                bag_embedding = embeddings[segment_idxs]  # [3, 128]
                self.samples.append((bag_embedding, label_idx, uid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y, uid = self.samples[idx]
        return x, torch.tensor(y).long(), torch.tensor(uid).long()


In [ ]:
mil_multi_dataset = MultiClassMILDataset(pose_embeddings_triplet, y_segments, user_segments)
num_classes = len(mil_multi_dataset.label_names)


In [ ]:
from torch.utils.data import DataLoader, Subset

def split_LOSO(dataset, test_user_id):
    train_idx, test_idx = [], []

    for i in range(len(dataset)):
        _, _, uid = dataset[i]
        if uid == test_user_id:
            test_idx.append(i)
        else:
            train_idx.append(i)

    return Subset(dataset, train_idx), Subset(dataset, test_idx)


In [ ]:
class MILTransformerMulticlass(nn.Module):
    def __init__(self, input_dim=128, model_dim=128, num_heads=4, num_layers=2, num_classes=8):
        super().__init__()
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True),
            num_layers=num_layers
        )
        self.attention_weights = nn.Linear(model_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(model_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):  # x: [B, 3, 128]
        x = self.transformer(x)  # [B, 3, 128]
        attn_score = self.attention_weights(x)             # [B, 3, 1]
        attn_weights = torch.softmax(attn_score, dim=1)    # [B, 3, 1]
        pooled = torch.sum(attn_weights * x, dim=1)        # [B, 128]
        out = self.classifier(pooled)                      # [B, num_classes]
        return out


In [ ]:
from sklearn.metrics import classification_report

def train_and_eval_multiclass_MIL(model, train_set, test_set, label_names, epochs=10, batch_size=64, device='cuda'):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y, _ in train_loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            loss = F.cross_entropy(out, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] Train Loss: {total_loss / len(train_loader):.4f}")

    # Evaluation
    model.eval()
    all_y_true = []
    all_y_pred = []

    with torch.no_grad():
        for x, y, _ in test_loader:
            x = x.to(device)
            logits = model(x)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            all_y_pred.extend(pred)
            all_y_true.extend(y.numpy())

    print("\n📊 Classification Report (8-class):")
    print(classification_report(all_y_true, all_y_pred, target_names=label_names))


In [ ]:
# Danh sách các user cần đánh giá
test_users = [1, 2, 3, 4, 5]

# Dictionary để lưu kết quả từng user
results = {}

for user_id in test_users:
    print(f"\n{'='*50}")
    print(f"🔁 Đang chạy LOSO cho user {user_id}")
    print(f"{'='*50}")

    # Phân chia dữ liệu
    train_set, test_set = split_LOSO(mil_multi_dataset, test_user_id=user_id)

    # Khởi tạo và huấn luyện mô hình
    mil_model_multi = MILTransformerMulticlass(num_classes=num_classes)

    # Huấn luyện và đánh giá
    print(f"\n📊 Kết quả cho user {user_id}:")
    train_and_eval_multiclass_MIL(
        mil_model_multi,
        train_set,
        test_set,
        label_names=mil_multi_dataset.label_names
    )

    # Lưu lại model nếu cần
    # torch.save(mil_model_multi.state_dict(), f'mil_model_user_{user_id}.pth')


🔁 Đang chạy LOSO cho user 1

📊 Kết quả cho user 1:
[Epoch 1] Train Loss: 0.3424
[Epoch 2] Train Loss: 0.2792
[Epoch 3] Train Loss: 0.2646
[Epoch 4] Train Loss: 0.2540
[Epoch 5] Train Loss: 0.2473
[Epoch 6] Train Loss: 0.2404
[Epoch 7] Train Loss: 0.2342
[Epoch 8] Train Loss: 0.2293
[Epoch 9] Train Loss: 0.2261
[Epoch 10] Train Loss: 0.2228

📊 Classification Report (8-class):
                 precision    recall  f1-score   support

      Attacking       0.47      0.59      0.52      1474
         Biting       0.97      0.97      0.97      1870
  Eating snacks       0.95      0.94      0.94      4910
   Head banging       0.98      0.98      0.98      1270
Sitting quietly       0.80      0.79      0.80      4770
Throwing things       0.90      0.78      0.84      1510
    Using phone       0.83      0.66      0.73      3660
        Walking       0.86      0.99      0.92      4859

       accuracy                           0.85     24323
      macro avg       0.85      0.84      0.84   

In [ ]:
from collections import defaultdict

uids = [1, 2, 3, 5]
dfs = []
for uid in uids:
    df = pd.read_csv(f'/content/drive/MyDrive/ISAS25/keypointlabel/keypoints_with_labels_{uid}.csv')
    df['user_id'] = uid
    dfs.append(df)

kp_all, label_all, frame_all, id_all = preprocess_all_users(dfs)
X_segments, y_segments, time_segments, user_segments = segment_all_users(kp_all, label_all, frame_all, id_all)

user_data = defaultdict(lambda: {"X": [], "y": [], "uid": []})
for x, y, u in zip(X_segments, y_segments, user_segments):
    user_data[u]["X"].append(x)
    user_data[u]["y"].append(y)
    user_data[u]["uid"].append(u)


In [ ]:
X_train, y_train = [], []
for uid in user_data:
    X_train.extend(user_data[uid]['X'])
    y_train.extend(user_data[uid]['y'])

triplets = build_triplets(X_train, y_train, num_triplets=10000)
triplet_dataset = TripletPoseDataset(X_train, triplets)

encoder = PoseEncoder()
encoder = train_encoder_triplet(encoder, triplet_dataset, epochs=10)
torch.save(encoder.state_dict(), '/content/pose_encoder_final.pth')


[Epoch 1] Triplet Loss: 0.3498
[Epoch 2] Triplet Loss: 0.2088
[Epoch 3] Triplet Loss: 0.1536
[Epoch 4] Triplet Loss: 0.1056
[Epoch 5] Triplet Loss: 0.0774
[Epoch 6] Triplet Loss: 0.0841
[Epoch 7] Triplet Loss: 0.0436
[Epoch 8] Triplet Loss: 0.0428
[Epoch 9] Triplet Loss: 0.0311
[Epoch 10] Triplet Loss: 0.0331


In [ ]:
embeddings, labels, user_ids = [], [], []
for uid in user_data:
    embs = extract_embeddings_from_encoder(user_data[uid]['X'], encoder)
    embeddings.append(embs)
    labels.extend(user_data[uid]['y'])
    user_ids.extend([uid] * len(user_data[uid]['X']))

embeddings = torch.cat(embeddings, dim=0)
mil_dataset = MultiClassMILDataset(embeddings, labels, user_ids)

mil_model = MILTransformerMulticlass(num_classes=len(mil_dataset.label_names))
train_loader = DataLoader(mil_dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(mil_model.parameters(), lr=1e-3)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mil_model.to(device)

for epoch in range(10):
    mil_model.train()
    total_loss = 0
    for x, y, _ in train_loader:
        x, y = x.to(device), y.to(device)
        out = mil_model(x)
        loss = F.cross_entropy(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[Epoch {epoch+1}] MIL Loss: {total_loss / len(train_loader):.4f}")


[Epoch 1] MIL Loss: 0.1953
[Epoch 2] MIL Loss: 0.1436
[Epoch 3] MIL Loss: 0.1297
[Epoch 4] MIL Loss: 0.1230
[Epoch 5] MIL Loss: 0.1253
[Epoch 6] MIL Loss: 0.1162
[Epoch 7] MIL Loss: 0.1072
[Epoch 8] MIL Loss: 0.1064
[Epoch 9] MIL Loss: 0.1066
[Epoch 10] MIL Loss: 0.1038


In [ ]:
class PredictionSmoother:
    def __init__(self, window_size=3):
        self.window = []
        self.window_size = window_size

    def smooth(self, pred):
        self.window.append(pred)
        if len(self.window) > self.window_size:
            self.window.pop(0)
        return Counter(self.window).most_common(1)[0][0]

In [ ]:
df_test = pd.read_csv('/content/drive/MyDrive/ISAS25/keypointlabel/test_data_keypoint.csv')
df_test['Action Label'] = 'None'  # Tạm gán để reuse pipeline

# Chuẩn hóa + segment
kp_test = normalize_pose_keypoints(df_test)
frames_test = df_test['frame_id'].values
test_segments, _, test_frame_ids = sliding_window_segments(kp_test, ['None'] * len(kp_test), frames_test)

# Load encoder
encoder.load_state_dict(torch.load('/content/pose_encoder_final.pth'))
encoder.eval()
test_embeddings = extract_embeddings_from_encoder(test_segments, encoder, device=device)

# Gom segment thành bag
bags = []
segment_length = 3
for i in range(0, len(test_embeddings) - segment_length + 1):
    bag = test_embeddings[i:i + segment_length]
    bags.append(bag)

# Dự đoán nhãn từng bag
mil_model.eval()
predictions = []
with torch.no_grad():
    for bag in bags:
        x = bag.unsqueeze(0).to(device)
        out = mil_model(x)
        pred = torch.argmax(out, dim=1).item()
        predictions.append(pred)

# Gán nhãn cho các frame đầu của 3 segment trong mỗi bag
label_names = mil_dataset.label_names
smoother = PredictionSmoother(window_size=3)
frame_to_label = {}

for i, pred in enumerate(predictions):
    smoothed = smoother.smooth(label_names[pred])
    segment_frame_ids = test_frame_ids[i:i + segment_length]
    for fid in segment_frame_ids:
        frame_to_label[fid] = smoothed

# Gán nhãn vào từng dòng trong dataframe
df_test['Predicted Label'] = df_test['frame_id'].map(frame_to_label)

# 📌 Bổ sung: Fill nhãn cho các frame chưa có bằng nhãn gần nhất phía trên
df_test['Predicted Label'] = df_test['Predicted Label'].fillna(method='ffill')  # forward fill
df_test['Predicted Label'] = df_test['Predicted Label'].fillna(method='bfill')  # nếu đầu file vẫn null, fill ngược

# Lưu kết quả
df_test[['frame_id', 'Predicted Label']].to_csv('/content/drive/MyDrive/ISAS25/keypointlabel/predictions_test.csv', index=False)
print("✅ Đã gán nhãn đầy đủ cho mọi dòng và lưu kết quả.")


✅ Đã gán nhãn đầy đủ cho mọi dòng và lưu kết quả.


/tmp/ipython-input-19-2899682606.py:46: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_test['Predicted Label'] = df_test['Predicted Label'].fillna(method='ffill')  # forward fill
/tmp/ipython-input-19-2899682606.py:47: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_test['Predicted Label'] = df_test['Predicted Label'].fillna(method='bfill')  # nếu đầu file vẫn null, fill ngược


In [ ]:
import pandas as pd

# Đọc file raw keypoint test
df_keypoint = pd.read_csv('/content/drive/MyDrive/ISAS25/keypointlabel/test_data_keypoint.csv')

# Đọc file nhãn đã dự đoán
df_label = pd.read_csv('/content/drive/MyDrive/ISAS25/keypointlabel/predictions_test.csv')

# Gộp lại theo frame_id
df_merged = pd.merge(df_keypoint, df_label, on='frame_id', how='left')

# Kiểm tra nhanh
print(df_merged[['frame_id', 'Predicted Label']].head())

# Lưu lại
df_merged.to_csv('/content/drive/MyDrive/ISAS25/keypointlabel/test_data_with_predicted_labels.csv', index=False)
print("✅ Đã lưu file có nhãn kèm keypoint.")


   frame_id  Predicted Label
0         0  Sitting quietly
1         1  Sitting quietly
2         2  Sitting quietly
3         3  Sitting quietly
4         4  Sitting quietly
✅ Đã lưu file có nhãn kèm keypoint.
